Deep Learning Embeddings Roadmap: Co-occurrence ->
Word2Vec -> GloVe
Phase 1: Co-occurrence Matrix
Goal: Understand the raw statistics behind word embeddings.

1. Build a small corpus manually (10–20 sentences).
2. Construct the co-occurrence matrix X, where X_ij = count of word j in context of word
   i.
3. Normalize rows to get probabilities P(j | i).
4. Observe that words appearing in similar contexts have similar row vectors.
   Phase 2: Word2Vec (Skip-gram / CBOW)
   Goal: See how predictive models turn raw counts into dense embeddings.
5. Implement skip-gram on your small corpus.
6. Optional: Add negative sampling to approximate softmax.
7. Train for a few epochs and extract learned embeddings.
8. Compute cosine similarity between words.
9. Compare embeddings with raw co-occurrence vectors to see compression into lower
   dimensions.
   Phase 3: Analogy and Similarity Tasks
   Goal: See emergent semantics from embeddings.
10. Test word relationships: e.g., "king - man + woman".
11. Observe how skip-gram captures semantic relationships unlike raw co-occurrence
    matrices.
    Phase 4: GloVe (Global Vectors)
    Goal: Bridge count-based and predictive embeddings.
12. Understand GloVe objective: J = sum_ij f(X_ij) (w_i^T w_j + b_i + b_j - log X_ij)^2.
13. Use your co-occurrence matrix from Phase 1.
14. Implement simple matrix factorization (SVD) on log(X).
15. Compare embeddings to skip-gram embeddings.
16. Observe that both capture semantic similarity via different mechanisms (predictive
    vs count-based).
    Phase 5: Visualization & Intuition Check
17. Reduce embeddings to 2D using PCA/t-SNE.
18. Plot words like "cat", "dog", "king", "queen".
19. Compare raw co-occurrence vectors, skip-gram embeddings, and GloVe embeddings.
20. Insight: embeddings encode context similarity, which produces semantics.
    Outcome After These Experiments

- Understand math behind embeddings (raw counts -> predictive -> global).
- See embeddings emerge visually.
- Understand why Word2Vec and GloVe work and how they relate.
- Ready for contextual embeddings (ELMo, BERT) and transformers with full
  understanding.


way to do the nlp


- do the cooccurance matrix -- with 1 if present if not 0 ###not gona do it due to it is not feasible practically

- the co occurance matrix with the PMI just construct with small sample

  - two ways if less then zero then zero
  - positive pmi

- dimension reduction using svd


In [ ]:
import numpy as np
import nltk
from scipy.sparse.linalg import svds
import sys
nltk.download('brown')

from nltk.corpus import brown



In [ ]:
# constants

WINDOW_SIZE: int=3

k:int= 50

In [ ]:
sentences=brown.sents()[:100]

corpus = [sentence[i:i+WINDOW_SIZE] for sentence in sentences for i in range(len(sentence)-WINDOW_SIZE+1)]

print(corpus)

In [ ]:
def PPMI(cooccurance,N):
    '''
        PMI(w, c) = log p(c|w)
        p(c)
        = log |  count(w, c) ∗ N    |
              |-------------------  |
              |count(c) ∗ count(w)  |
    '''
    countWords = np.sum(cooccurance, axis=1)
    
    print(countWords)
    
    with np.errstate(divide='ignore', invalid='ignore'):
        expected = np.outer(countWords, countWords) / N
        pmi = np.log10(cooccurance / expected)    
        pmi[np.isnan(pmi)] = 0
        pmi[pmi < 0] = 0
        
    cooccurance[:, :] = pmi
    return cooccurance



In [ ]:

words=sorted(set(word for sentence in sentences for word in sentence))

word2id = {w: i for i, w in enumerate(words)}
id2word = {i: w for w, i in word2id.items()}

#calculating the coocurance matrix

cooccurance=np.zeros((len(words),len(words)),dtype=float)

for window in corpus:
    for i, word in enumerate(window):
        w_idx = word2id[word]

        for j, neighbor in enumerate(window):
            if i != j:
                n_idx = word2id[neighbor]
                cooccurance[w_idx, n_idx] += 1
np.set_printoptions(threshold=sys.maxsize)

print("Co-occurrence sum:", len(corpus),np.sum(cooccurance))

In [ ]:
# calculating the ppmi mat
ppmi_mat=PPMI(cooccurance,len(words))

print(ppmi_mat)



In [ ]:
### doing the svd (singular value decomposition on it)

def svd(matrix):
    
    U, S, Vt = svds(matrix,k=k)

    return [U[:, ::-1], S[::-1], Vt[::-1, :]]
    

svd_ppmi=svd(ppmi_mat)

for i in svd_ppmi:
    print(np.shape(i))
    
### and then doing the approximation

Word_embeding = svd_ppmi[0] @ np.diag(svd_ppmi[1])

print(f"the size of the new Embeding is {np.shape(Word_embeding)}")

In [ ]:
# print(Word_embeding)

print(words)
print(np.dot(Word_embeding[word2id['Fulton']],Word_embeding[word2id['County']].T))

# Continious Bag of word

In [52]:
# datasets
np_corpus = np.array(corpus)
print(np.shape(np_corpus))

X_train = np_corpus[:int(0.7*np.shape(np_corpus)[0]), 0 : WINDOW_SIZE-1]
Y_train = np_corpus[:int(0.7*np.shape(np_corpus)[0]) , WINDOW_SIZE-1:WINDOW_SIZE]

X_test = np_corpus[int(0.7*np.shape(np_corpus)[0])  : , 0 : WINDOW_SIZE-1]
Y_test = np_corpus[int(0.7*np.shape(np_corpus)[0])  : , WINDOW_SIZE-1:WINDOW_SIZE]


print(np.shape(X_train),np.shape(Y_train))
print(np.shape(X_test),np.shape(Y_test))


(2069, 3)
(1448, 2) (1448, 1)
(621, 2) (621, 1)


In [ ]:
# constants

L=3 #is the no if layer

MAX_ITR=100 

LEARNING_RATE=0.5 #η 

NO_OF_INPUT=[(WINDOW_SIZE-1)*len(X_train[0]),k,len(Y_train[0])] # mem friendly input h1,h2 output

NO_OF_OUTPUT=10

BATCH_SIZE=10000

In [27]:
# Continuous bag of word exeqution

